In [5]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2
import sys

In [ ]:
def solve():
    input = sys.stdin.readline
    n, Q = map(int, input().split())

    dist = []
    num_vehicles = 1
    depot = 0
    demands = [0] * (2 * n + 1) # Mảng nhu cầu, đón khách = 1, trả khách = -1, depot = 0
    for i in range(1, n + 1):
        demands[i] = 1
        demands[i + n] = -1
    
    for i in range(2 * n + 1):
        dist.append(list(map(int, input().split())))

    manager = pywrapcp.RoutingIndexManager(len(dist), num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return dist[from_node][to_node]

    transit_distance_callback = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_distance_callback)

    routing.AddDimension(
        transit_distance_callback,
        0,
        3000000,
        True,
        'Distance'
    )
    distance_dimension = routing.GetDimensionOrDie('Distance')

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return demands[from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        [Q],
        True,
        'Capacity'
    )

    pickup_deliveries = [[i, i + n] for i in range(1, n + 1)]
    for request in pickup_deliveries:
        pickup_index = manager.NodeToIndex(request[0])
        delivery_index = manager.NodeToIndex(request[1])
        routing.AddPickupAndDelivery(pickup_index, delivery_index)

        routing.solver().Add(
            distance_dimension.CumulVar(pickup_index) <= distance_dimension.CumulVar(delivery_index)
        )
        routing.solver().Add(
            routing.VehicleVar(pickup_index) == routing.VehicleVar(delivery_index)
        )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.FromSeconds(5)
    solution = routing.SolveWithParameters(search_parameters)
    
    status = routing.status()
    if status != 1:
        print(-1)
        return

    if solution:
        print(solution.ObjectiveValue())
        index = routing.Start(0)
        route = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(str(node))
            index = solution.Value(routing.NextVar(index))
        end_node = manager.IndexToNode(index)
        route.append(str(end_node))
        print(" ".join(route))
    else:
        print(-1)

In [8]:
if __name__ == "__main__":
    f = open("input.txt", "r")
    sys.stdin = f
    solve()
    f.close()

8
0 1 2 4 3 0
